# ขั้นตอนการจัดการข้อมูลอสังหาริมทรัพย์ — Real Estate Aggregator

Notebook นี้รวบรวมโค้ดและอธิบายขั้นตอนจาก repository [chanidapa14092547/real-estate-aggregator](https://github.com/chanidapa14092547/real-estate-aggregator) โดยอ่านจาก commit `cf4c4affdc43d3f1450fe67efcd8185c224a72c4` เมื่อ 6 ตุลาคม 2026

ข้อมูลที่ pipeline นี้ใช้เป็น **ข้อมูลจำลอง (mock data)** ไม่ใช่ประกาศจริงที่ scrape มาจาก DDProperty, Baania หรือ HipFlat ส่วน `scraper_template.py` เป็นตัวอย่างโครงสร้าง scraper เท่านั้น

ลำดับงาน: สร้างข้อมูลจำลอง → รวม schema → ลบข้อมูลซ้ำ → จัดการค่าว่าง → กรองสแปมและแก้ค่าผิด → แปลงราคา/พื้นที่/ทำเล → สร้าง features → คัดกรองข้อมูลสุดท้าย → เตรียมข้อมูล ML → ส่งต่อ dashboard

### วิธีใช้งาน

1. เปิดไฟล์นี้ใน VS Code (Python + Jupyter extensions), Jupyter หรือ Google Colab
2. หากยังไม่มี dependencies ให้ติดตั้ง `pandas>=2.0 numpy scikit-learn joblib` ใน environment ของ notebook
3. รันเซลล์จากบนลงล่าง โค้ดหลักอยู่ในไฟล์นี้ครบ ไม่ต้อง clone repo หรือดาวน์โหลด CSV ก่อน
4. ค่าเริ่มต้นสร้างข้อมูลใหม่ด้วย `seed=42` ในโฟลเดอร์ `real_estate_notebook_run` หากต้องการใช้ CSV ที่มีอยู่ ให้ตั้ง `GENERATE_MOCK=False` แล้ววางไฟล์ทั้งสามใน `data/raw` ภายใต้โฟลเดอร์นั้น
5. ผลลัพธ์ที่แนบในไฟล์เป็นผลจากการรันทดสอบ notebook นี้ ตัวเลขจาก report เดิมใน repo แสดงแยกไว้ท้ายไฟล์

คำอธิบายและตารางตรวจสอบถูกเพิ่มเพื่อช่วยอ่าน ส่วน generator, cleaner, transformer และ predictor คงนิยามจาก source ของ repo; แยกการเรียกใช้แต่ละ method ให้เห็นข้อมูลระหว่างขั้น ไม่เปลี่ยนกฎการจัดการข้อมูลเดิม


In [1]:
from pathlib import Path
import os, json, time, platform
import pandas as pd
import numpy as np
import sklearn
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)

RUN_DIR = Path.cwd() / 'real_estate_notebook_run'
GENERATE_MOCK = True
TRAIN_MODELS = True
for folder in ['data/raw', 'data/cleaned', 'data/transformed', 'models']:
    (RUN_DIR / folder).mkdir(parents=True, exist_ok=True)
report = {}
audit = []
pipeline_started = time.perf_counter()
def record_stage(name, before, after):
    audit.append({'step': name, 'before': int(before), 'after': int(after), 'removed': int(before-after)})
def profile(df):
    return pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'missing_pct': (df.isna().mean()*100).round(2)})
print('Output directory:', RUN_DIR)
print('Versions:', {'python': platform.python_version(), 'pandas': pd.__version__, 'numpy': np.__version__, 'sklearn': sklearn.__version__})

Output directory: /workspace/scratch/3f951e18699d/real_estate_notebook_run
Versions: {'python': '3.12.14', 'pandas': '2.2.3', 'numpy': '2.3.5', 'sklearn': '1.8.0'}


## 1. สร้างและตรวจข้อมูลดิบ

Generator ใช้ Python random และ NumPy seed=42 สร้างข้อมูลพื้นฐาน 17,000 + 17,000 + 16,000 แถว แล้วเพิ่มแถวซ้ำอีก 350 แถว รวม 50,350 แถว พร้อมใส่ปัญหาเพื่อทดลอง cleaning เช่น ค่าว่าง ราคาข้อความ ราคา/พื้นที่ติดลบ ค่า 0 ข้อมูลสลับคอลัมน์ สแปม และรูปแบบวันที่ต่างกัน

ราคาใน mock data ถูกสร้างจากพื้นที่ × ราคาฐานตามประเภท × ตัวคูณทำเล × noise และ bonus จากรถไฟฟ้า/ชั้นอาคาร วันที่สร้างอ้างอิง `datetime.now()` ดังนั้น seed เดิมไม่ได้ทำให้วันที่เหมือนกันทุกวัน

| แหล่งจำลอง | ไฟล์ | รูปแบบต่างกัน |
|---|---|---|
| DDProperty | ddproperty.csv | ชื่อคอลัมน์อังกฤษ ราคา THB พื้นที่ ตร.ม. |
| Baania | baania.csv | ชื่อคอลัมน์ไทย ราคาข้อความ พื้นที่ ตร.วา |
| HipFlat | hipflat.csv | cost_usd, area และ land_rai/ngan/wah ไม่มี location โดยตรง |

แหล่งโค้ด: [src/extraction/mock_generator.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/extraction/mock_generator.py)

In [2]:
import os
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta

def generate_mock_data(base_path: str, seed: int = 42):
    """Generates mock real estate data with injected problems."""
    np.random.seed(seed)
    random.seed(seed)
    
    raw_dir = os.path.join(base_path, 'data', 'raw')
    os.makedirs(raw_dir, exist_ok=True)
    
    # Common lists
    locations = [
        'สุขุมวิท', 'Sukhumvit', 'สีลม', 'Silom', 'สาทร', 'Sathorn', 'ลาดพร้าว', 'Lat Phrao',
        'พระราม9', 'Rama 9', 'อ่อนนุช', 'On Nut', 'บางนา', 'Bang Na', 'รังสิต', 'Rangsit',
        'ธนบุรี', 'Thonburi', 'พญาไท', 'Phaya Thai', 'อารีย์', 'Ari', 'ทองหล่อ', 'Thong Lo',
        'เอกมัย', 'Ekkamai', 'ราชเทวี', 'Ratchathewi', 'ห้วยขวาง', 'Huai Khwang',
        'บางกะปิ', 'Bang Kapi', 'มีนบุรี', 'Min Buri', 'หนองจอก', 'Nong Chok',
        'ตลิ่งชัน', 'Taling Chan', 'บางแค', 'Bang Khae', 'จตุจักร', 'Chatuchak'
    ]
    prop_types = ['คอนโด', 'Condo', 'condominium', 'บ้านเดี่ยว', 'House', 'Single House', 'detached house', 'ทาวน์โฮม', 'Townhome', 'Townhouse', 'ที่ดิน', 'Land', 'land plot']
    
    def random_date(start_days_ago=365):
        base_date = datetime.now() - timedelta(days=random.randint(0, start_days_ago))
        return base_date
        
    # Zone multiplier for price-per-sqm (CBD is most expensive)
    zone_price_multiplier = {
        'สุขุมวิท': 1.8, 'Sukhumvit': 1.8, 'สีลม': 1.7, 'Silom': 1.7,
        'สาทร': 1.65, 'Sathorn': 1.65, 'ทองหล่อ': 2.0, 'Thong Lo': 2.0,
        'เอกมัย': 1.6, 'Ekkamai': 1.6, 'อารีย์': 1.5, 'Ari': 1.5,
        'พญาไท': 1.3, 'Phaya Thai': 1.3, 'ราชเทวี': 1.25, 'Ratchathewi': 1.25,
        'ลาดพร้าว': 1.1, 'Lat Phrao': 1.1, 'พระราม9': 1.2, 'Rama 9': 1.2,
        'อ่อนนุช': 1.05, 'On Nut': 1.05, 'ห้วยขวาง': 1.1, 'Huai Khwang': 1.1,
        'จตุจักร': 1.15, 'Chatuchak': 1.15, 'บางนา': 0.85, 'Bang Na': 0.85,
        'บางกะปิ': 0.8, 'Bang Kapi': 0.8, 'ธนบุรี': 0.75, 'Thonburi': 0.75,
        'ตลิ่งชัน': 0.7, 'Taling Chan': 0.7, 'บางแค': 0.65, 'Bang Khae': 0.65,
        'รังสิต': 0.55, 'Rangsit': 0.55, 'มีนบุรี': 0.5, 'Min Buri': 0.5,
        'หนองจอก': 0.4, 'Nong Chok': 0.4
    }
    
    # Base price-per-sqm by property type (THB)
    type_base_price = {
        'คอนโด': 120000, 'Condo': 120000, 'condominium': 120000,
        'บ้านเดี่ยว': 80000, 'House': 80000, 'Single House': 80000, 'detached house': 80000,
        'ทาวน์โฮม': 65000, 'Townhome': 65000, 'Townhouse': 65000,
        'ที่ดิน': 45000, 'Land': 45000, 'land plot': 45000
    }
    
    bts_stations = [
        'BTS สุขุมวิท', 'BTS สีลม', 'BTS อ่อนนุช', 'BTS เอกมัย', 'BTS ทองหล่อ',
        'BTS พร้อมพงษ์', 'BTS อารีย์', 'BTS สะพานควาย', 'BTS ชิดลม',
        'MRT ลาดพร้าว', 'MRT พระราม9', 'MRT ห้วยขวาง', 'MRT สุขุมวิท',
        'MRT จตุจักร', 'MRT บางซื่อ', 'MRT ศูนย์วัฒนธรรม'
    ]
    
    def generate_base_listing(source_name, id_prefix, p_type):
        loc = random.choice(locations)
        title = f"{p_type} for sale at {loc}"
        desc = f"Beautiful {p_type} located in {loc}. Great view."
        
        # Area based on property type
        if p_type in ['คอนโด', 'Condo', 'condominium']:
            area = random.uniform(25, 200)
            beds = max(1, int(area / 40) + random.randint(-1, 1))
            beds = min(beds, 5)
            baths = max(1, beds - random.randint(0, 1))
            floor = random.randint(1, 40)
        elif p_type in ['บ้านเดี่ยว', 'House', 'Single House', 'detached house']:
            area = random.uniform(100, 500)
            beds = max(2, int(area / 60) + random.randint(-1, 1))
            beds = min(beds, 8)
            baths = max(2, beds - random.randint(0, 2))
            floor = random.randint(1, 3)
        elif p_type in ['ทาวน์โฮม', 'Townhome', 'Townhouse']:
            area = random.uniform(60, 200)
            beds = max(2, int(area / 50) + random.randint(-1, 1))
            beds = min(beds, 5)
            baths = max(1, beds - random.randint(0, 1))
            floor = random.randint(2, 4)
        else:  # Land
            area = random.uniform(50, 5000)
            beds = 0
            baths = 0
            floor = 0
        
        # Price = area × base_price_per_sqm × zone_multiplier × noise
        base_ppsqm = type_base_price.get(p_type, 80000)
        zone_mult = zone_price_multiplier.get(loc, 1.0)
        noise = random.gauss(1.0, 0.15)  # ±15% random variation
        
        # BTS proximity bonus
        near_bts = random.choice(bts_stations) if random.random() < 0.6 else ''
        bts_bonus = 1.1 if near_bts else 1.0
        
        # Floor bonus for condos
        floor_bonus = 1.0 + (floor * 0.005) if p_type in ['คอนโด', 'Condo', 'condominium'] else 1.0
        
        price = area * base_ppsqm * zone_mult * noise * bts_bonus * floor_bonus
        price = max(price, 100000)  # Minimum price floor
        
        dt = random_date()
        
        return {
            'price': price,
            'area_sqm': area,
            'beds': beds,
            'baths': baths,
            'floor': floor,
            'loc': loc,
            'title': title,
            'desc': desc,
            'near_bts': near_bts,
            'dt': dt
        }

    print("Generating Source 1: DDProperty (17,000 records)...")
    dd_records = []
    for i in range(17000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("DDProperty", "DD", p_type)
        
        price = int(base['price'])
        area = base['area_sqm']
        beds = base['beds']
        
        # Inject problems
        r = random.random()
        if r < 0.08:
            price = np.nan
        if r < 0.05:
            area = np.nan
        if 0.13 < r < 0.16:
            price = -abs(price) if price is not np.nan else price
            area = -abs(area) if area is not np.nan else area
        if 0.16 < r < 0.18:
            price = 0
        if 0.18 < r < 0.20 and price is not np.nan and area is not np.nan:
            # Swap
            price, area = area, price
        if 0.20 < r < 0.22:
            beds = random.randint(21, 50)
            
        title = base['title']
        desc = base['desc']
        if r < 0.05: title += ' &amp; <br>'
        if r < 0.10: title += ' 🏠🔥⭐'
        if 0.10 < r < 0.15: title = f"คอนโดสวย โทร 081-{random.randint(100,999)}-{random.randint(1000,9999)} " + title
            
        record = {
            'id': f'DD{i}',
            'title': title,
            'description': desc,
            'price': price,
            'area_sqm': area,
            'bedrooms': beds,
            'bathrooms': base['baths'],
            'property_type': p_type,
            'location': base['loc'],
            'floor': base['floor'],
            'near_bts': base['near_bts'],
            'posted_date': base['dt'].strftime('%Y-%m-%d'),
            'url': f'https://ddproperty.fake/listing/{i}',
            'source': 'DDProperty'
        }
        dd_records.append(record)
        
    # Duplicate some rows
    for _ in range(200):
        idx = random.randint(0, len(dd_records)-1)
        dd_records.append(dd_records[idx].copy())
        
    pd.DataFrame(dd_records).to_csv(os.path.join(raw_dir, 'ddproperty.csv'), index=False)

    print("Generating Source 2: Baania (17,000 records)...")
    baania_records = []
    # Mix thai formats
    for i in range(17000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("Baania", "BN", p_type)
        
        price_val = base['price']
        price_str = f"{int(price_val)}"
        
        r = random.random()
        if r < 0.1: price_str = f"{price_val/1_000_000:.1f} ล้าน"
        elif r < 0.2: price_str = f"{int(price_val/100_000)} แสน"
        elif r < 0.3: price_str = f"{int(price_val):,} บาท"
        elif r < 0.35: price_str = "ราคาต่อรอง"
        elif r < 0.4: price_str = "สอบถาม"
        
        beds = base['beds']
        if r < 0.10: beds = np.nan
        
        area_wah = base['area_sqm'] / 4.0
        if r < 0.05: area_wah = 999999
        
        title = base['title']
        desc = base['desc']
        if r < 0.05: title = "ด่วน!!! " * 3 + title
        if 0.05 < r < 0.10: 
            desc = desc.upper() + " Call 088888888"
            desc = "รับฝากขาย-เช่า ติดต่อ... " + desc
            
        loc = base['loc']
        if loc == 'Sukhumvit' or loc == 'สุขุมวิท':
            loc = random.choice(['สุขุมวิท', 'สุขุมวิท ', 'Sukhumvit', ' สุขุมวิท', 'สุขุมวิทร์'])
            
        # Mixed dates
        dt_str = base['dt'].strftime('%Y-%m-%d')
        if r < 0.3: dt_str = base['dt'].strftime('%d/%m/%Y')
        elif r < 0.6: dt_str = base['dt'].strftime('%d %b %y')
            
        record = {
            'รหัส': f'BN{i}',
            'หัวข้อ': title,
            'รายละเอียด': desc,
            'ราคา': price_str,
            'พื้นที่_ตร.วา': area_wah,
            'ห้องนอน': beds,
            'ห้องน้ำ': base['baths'],
            'ประเภท': p_type,
            'ทำเล': loc,
            'ชั้น': base['floor'],
            'ใกล้รถไฟฟ้า': base['near_bts'],
            'วันที่ลง': dt_str,
            'ลิงก์': f'https://baania.fake/p/{i}',
            'แหล่ง': 'Baania'
        }
        baania_records.append(record)
        
    for _ in range(150):
        idx = random.randint(0, len(baania_records)-1)
        dup = baania_records[idx].copy()
        dup['ราคา'] = str(dup['ราคา']) + ' '
        baania_records.append(dup)

    pd.DataFrame(baania_records).to_csv(os.path.join(raw_dir, 'baania.csv'), index=False)
    
    print("Generating Source 3: HipFlat (16,000 records)...")
    hipflat_records = []
    for i in range(16000):
        p_type = random.choice(prop_types)
        base = generate_base_listing("HipFlat", "HF", p_type)
        
        cost_usd = base['price'] / 35.0
        
        r = random.random()
        if r < 0.05: cost_usd = base['price'] # accidentaly in THB
        if 0.05 < r < 0.08: cost_usd = random.choice([0, 1])
        
        sqm = base['area_sqm']
        land_rai = sqm // 1600
        land_ngan = (sqm % 1600) // 400
        land_wah = ((sqm % 1600) % 400) / 4.0
        
        if r < 0.07:
            land_rai, land_ngan, land_wah = np.nan, np.nan, np.nan
        elif 0.07 < r < 0.12:
            land_rai += 0.5 # Fractional rai double counting
            
        type_val = p_type
        if r < 0.03: type_val = random.choice(['asdf', '123', ''])
        
        beds_val = base['beds']
        if r < 0.05: beds_val = 'studio'
        elif 0.05 < r < 0.1: beds_val = '3+1'
        elif 0.1 < r < 0.15: beds_val = 'N/A'
        
        dt_str = base['dt'].strftime('%Y-%m-%d')
        if r < 0.3: dt_str = base['dt'].strftime('%b %d, %Y')
        elif r < 0.6: dt_str = base['dt'].strftime('%d/%m/%y')
        
        name = base['title']
        if r < 0.05: name += '\u0e00'
        
        record = {
            'listing_id': f'HF{i}',
            'name': name,
            'desc': base['desc'],
            'cost_usd': cost_usd,
            'land_rai': land_rai,
            'land_ngan': land_ngan,
            'land_wah': land_wah,
            'beds': beds_val,
            'baths': base['baths'],
            'type': type_val,
            'area': sqm,
            'level': base['floor'],
            'date': dt_str,
            'platform': 'HipFlat'
        }
        hipflat_records.append(record)

    pd.DataFrame(hipflat_records).to_csv(os.path.join(raw_dir, 'hipflat.csv'), index=False)
    print("Done generating mock data.")

In [3]:
t0 = time.perf_counter()
if GENERATE_MOCK:
    generate_mock_data(str(RUN_DIR), seed=42)
    report['generation_time'] = time.perf_counter() - t0
else:
    print('Use existing raw CSV files')
raw_tables = {}
for name in ['ddproperty', 'baania', 'hipflat']:
    path = RUN_DIR / 'data/raw' / f'{name}.csv'
    if not path.exists():
        raise FileNotFoundError(f'Place the required raw CSV here: {path}')
    raw_tables[name] = pd.read_csv(path)
    print(name, raw_tables[name].shape, list(raw_tables[name].columns))
    display(raw_tables[name].head(3))
display(pd.DataFrame({name: df.isna().sum() for name,df in raw_tables.items()}).fillna(0))

Generating Source 1: DDProperty (17,000 records)...
Generating Source 2: Baania (17,000 records)...
Generating Source 3: HipFlat (16,000 records)...
Done generating mock data.
ddproperty (17200, 14) ['id', 'title', 'description', 'price', 'area_sqm', 'bedrooms', 'bathrooms', 'property_type', 'location', 'floor', 'near_bts', 'posted_date', 'url', 'source']
baania (17150, 14) ['รหัส', 'หัวข้อ', 'รายละเอียด', 'ราคา', 'พื้นที่_ตร.วา', 'ห้องนอน', 'ห้องน้ำ', 'ประเภท', 'ทำเล', 'ชั้น', 'ใกล้รถไฟฟ้า', 'วันที่ลง', 'ลิงก์', 'แหล่ง']
hipflat (16000, 14) ['listing_id', 'name', 'desc', 'cost_usd', 'land_rai', 'land_ngan', 'land_wah', 'beds', 'baths', 'type', 'area', 'level', 'date', 'platform']


,id,title,description,price,area_sqm,bedrooms,bathrooms,property_type,location,floor,near_bts,posted_date,url,source
0,DD0,ที่ดิน for sale at Lat Phrao,Beautiful ที่ดิน located in Lat Phrao. Great view.,8.459608e+06,1.738032e+02,0,0,ที่ดิน,Lat Phrao,0,NaN,2025-10-25,https://ddproperty.fake/listing/0,DDProperty
1,DD1,Townhome for sale at Sathorn,Beautiful Townhome located in Sathorn. Great view.,1.426690e+02,1.860367e+07,2,2,Townhome,Sathorn,2,BTS สุขุมวิท,2025-12-23,https://ddproperty.fake/listing/1,DDProperty
2,DD2,ที่ดิน for sale at หนองจอก,Beautiful ที่ดิน located in หนองจอก. Great view.,3.387698e+07,2.126623e+03,0,0,ที่ดิน,หนองจอก,0,NaN,2026-07-17,https://ddproperty.fake/listing/2,DDProperty


,รหัส,หัวข้อ,รายละเอียด,ราคา,พื้นที่_ตร.วา,ห้องนอน,ห้องน้ำ,ประเภท,ทำเล,ชั้น,ใกล้รถไฟฟ้า,วันที่ลง,ลิงก์,แหล่ง
0,BN0,Condo for sale at Thonburi,Beautiful Condo located in Thonburi. Great view.,19083481,48.433730,3.0,2,Condo,Thonburi,8,BTS อ่อนนุช,25 May 26,https://baania.fake/p/0,Baania
1,BN1,ทาวน์โฮม for sale at Bang Na,Beautiful ทาวน์โฮม located in Bang Na. Great view.,"7,471,712 บาท",27.668385,2.0,1,ทาวน์โฮม,Bang Na,2,BTS ทองหล่อ,06/11/2025,https://baania.fake/p/1,Baania
2,BN2,ด่วน!!! ด่วน!!! ด่วน!!! Townhome for sale at Bang Kapi,Beautiful Townhome located in Bang Kapi. Great view.,4.3 ล้าน,999999.000000,NaN,2,Townhome,Bang Kapi,4,MRT สุขุมวิท,15/08/2026,https://baania.fake/p/2,Baania


,listing_id,name,desc,cost_usd,land_rai,land_ngan,land_wah,beds,baths,type,area,level,date,platform
0,HF0,Condo for sale at สาทร,Beautiful Condo located in สาทร. Great view.,448905.269858,0.0,0.0,16.647493,1,1,Condo,66.589970,1,21/07/26,HipFlat
1,HF1,Condo for sale at Sukhumvit,Beautiful Condo located in Sukhumvit. Great view.,1.000000,0.5,0.0,12.906964,3+1,1,Condo,51.627856,7,"Apr 06, 2026",HipFlat
2,HF2,คอนโด for sale at ธนบุรี,Beautiful คอนโด located in ธนบุรี. Great view.,434183.274589,0.0,0.0,43.827213,3,3,คอนโด,175.308853,6,2025-12-27,HipFlat


,ddproperty,baania,hipflat
area,0.0,0.0,0.0
area_sqm,839.0,0.0,0.0
bathrooms,0.0,0.0,0.0
baths,0.0,0.0,0.0
bedrooms,0.0,0.0,0.0
beds,0.0,0.0,811.0
cost_usd,0.0,0.0,0.0
date,0.0,0.0,0.0
desc,0.0,0.0,0.0
description,0.0,0.0,0.0


## 2. รวม schema และทำความสะอาด

นิยาม `DataCleaner` ด้านล่างเก็บ methods ต้นฉบับครบ หลังจากนิยามแล้วจะเรียกทีละขั้น เพื่อดูจำนวนแถวและค่าว่างระหว่างทาง

แหล่งโค้ด: [src/cleaning/cleaner.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/cleaning/cleaner.py)

In [4]:
import os
import pandas as pd
import numpy as np
import re

class DataCleaner:
    """Handles cleaning of real estate listings data."""
    
    def __init__(self, raw_dir, clean_dir):
        self.raw_dir = raw_dir
        self.clean_dir = clean_dir
        self.stats = {'before': 0, 'after': 0, 'operations': {}}
        
    def _log_op(self, op_name, count_diff):
        self.stats['operations'][op_name] = self.stats['operations'].get(op_name, 0) + count_diff
        print(f"  [{op_name}] Removed {count_diff} records")

    def _extract_location_from_title(self, title):
        """Extract location name from title text like 'Condo for sale at สุขุมวิท'."""
        if pd.isna(title):
            return np.nan
        locations = [
            'สุขุมวิท', 'Sukhumvit', 'สีลม', 'Silom', 'สาทร', 'Sathorn',
            'ลาดพร้าว', 'Lat Phrao', 'พระราม9', 'Rama 9', 'อ่อนนุช', 'On Nut',
            'บางนา', 'Bang Na', 'รังสิต', 'Rangsit', 'ธนบุรี', 'Thonburi',
            'พญาไท', 'Phaya Thai', 'อารีย์', 'Ari', 'ทองหล่อ', 'Thong Lo',
            'เอกมัย', 'Ekkamai', 'ราชเทวี', 'Ratchathewi', 'ห้วยขวาง', 'Huai Khwang',
            'บางกะปิ', 'Bang Kapi', 'มีนบุรี', 'Min Buri', 'หนองจอก', 'Nong Chok',
            'ตลิ่งชัน', 'Taling Chan', 'บางแค', 'Bang Khae', 'จตุจักร', 'Chatuchak'
        ]
        title_lower = str(title).lower()
        for loc in locations:
            if loc.lower() in title_lower:
                return loc
        return np.nan

    def load_and_merge(self):
        """Load 3 CSVs and merge into a unified schema."""
        # 1. DDProperty
        dd = pd.read_csv(os.path.join(self.raw_dir, 'ddproperty.csv'))
        dd_clean = pd.DataFrame({
            'id': dd['id'],
            'title': dd['title'],
            'description': dd['description'],
            'price_raw': dd['price'],
            'area_raw': dd['area_sqm'],
            'area_unit': 'sqm',
            'bedrooms': dd['bedrooms'],
            'bathrooms': dd['bathrooms'],
            'property_type': dd['property_type'],
            'location': dd['location'],
            'floor': dd['floor'],
            'near_bts': dd['near_bts'],
            'posted_date': dd['posted_date'],
            'source': dd['source']
        })
        
        # 2. Baania
        bn = pd.read_csv(os.path.join(self.raw_dir, 'baania.csv'))
        bn_clean = pd.DataFrame({
            'id': bn['รหัส'],
            'title': bn['หัวข้อ'],
            'description': bn['รายละเอียด'],
            'price_raw': bn['ราคา'],
            'area_raw': bn['พื้นที่_ตร.วา'],
            'area_unit': 'sqw',
            'bedrooms': bn['ห้องนอน'],
            'bathrooms': bn['ห้องน้ำ'],
            'property_type': bn['ประเภท'],
            'location': bn['ทำเล'],
            'floor': bn['ชั้น'],
            'near_bts': bn['ใกล้รถไฟฟ้า'],
            'posted_date': bn['วันที่ลง'],
            'source': bn['แหล่ง']
        })
        
        # 3. HipFlat — extract location from title since no location column
        hf = pd.read_csv(os.path.join(self.raw_dir, 'hipflat.csv'))
        hf_locations = hf['name'].apply(self._extract_location_from_title)
        
        hf_clean = pd.DataFrame({
            'id': hf['listing_id'],
            'title': hf['name'],
            'description': hf['desc'],
            'price_raw': hf['cost_usd'],
            'area_raw': hf['area'],
            'area_unit': 'sqm_split',
            'bedrooms': hf['beds'],
            'bathrooms': hf['baths'],
            'property_type': hf['type'],
            'location': hf_locations,
            'floor': hf['level'],
            'near_bts': pd.Series([np.nan]*len(hf)),
            'posted_date': hf['date'],
            'source': hf['platform'],
            'land_rai': hf['land_rai'],
            'land_ngan': hf['land_ngan'],
            'land_wah': hf['land_wah']
        })
        
        df = pd.concat([dd_clean, bn_clean, hf_clean], ignore_index=True)
        self.stats['before'] = len(df)
        print(f"  Loaded {len(dd_clean)} DDProperty + {len(bn_clean)} Baania + {len(hf_clean)} HipFlat = {len(df)} total")
        return df
        
    def deduplicate(self, df):
        """Remove duplicates."""
        initial_len = len(df)
        # Exact duplicates by id
        df = df.drop_duplicates(subset=['id'], keep='first')
        
        # Fuzzy/Cross-source duplicates: same title + location + similar price
        df = df.drop_duplicates(subset=['title', 'location', 'price_raw'], keep='first')
        
        self._log_op('deduplication', initial_len - len(df))
        return df
        
    def handle_missing(self, df):
        """Handle missing values appropriately."""
        initial_len = len(df)
        
        # Drop if both price and area missing
        mask = df['price_raw'].isna() & df['area_raw'].isna()
        df = df[~mask].copy()
        
        # Flag missing prices for imputation
        df['price_imputed'] = df['price_raw'].isna()
        
        # Drop untransformable text prices
        untransformable = ['ราคาต่อรอง', 'สอบถาม']
        df = df[~df['price_raw'].astype(str).str.strip().isin(untransformable)].copy()
        
        # Handle string bedrooms
        df['bedrooms'] = df['bedrooms'].replace({'studio': 0, '3+1': 4, 'N/A': np.nan})
        df['bedrooms'] = pd.to_numeric(df['bedrooms'], errors='coerce')
        
        # Infer bedrooms by property type
        condo_mask = (df['property_type'].str.contains('คอนโด|condo|condominium', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[condo_mask, 'bedrooms'] = 1
        house_mask = (df['property_type'].str.contains('บ้าน|house|detached', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[house_mask, 'bedrooms'] = 3
        townhome_mask = (df['property_type'].str.contains('ทาวน์|town', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[townhome_mask, 'bedrooms'] = 2
        land_mask = (df['property_type'].str.contains('ที่ดิน|land', case=False, na=False)) & df['bedrooms'].isna()
        df.loc[land_mask, 'bedrooms'] = 0
        
        self._log_op('missing_values', initial_len - len(df))
        return df

    def filter_spam_outliers(self, df):
        """Filter out spam, outliers and fix artifacts."""
        initial_len = len(df)
        
        # === TEXT CLEANUP ===
        # HTML/Emoji artifacts in title
        df['title'] = df['title'].astype(str).str.replace(r'&amp;|<br>|\\n\\t', '', regex=True)
        df['title'] = df['title'].str.replace(r'[🏠🔥⭐\u0e00]', '', regex=True)
        
        # === SPAM REMOVAL ===
        # Phone numbers in title
        phone_mask = df['title'].str.contains(r'0\d{1,2}[-\s]?\d{3}[-\s]?\d{4}', regex=True, na=False)
        df = df[~phone_mask].copy()
        
        # Agent ads in description
        agent_keywords = r'รับฝากขาย|รับฝากเช่า|นายหน้า'
        agent_mask = df['description'].astype(str).str.contains(agent_keywords, regex=True, na=False)
        df = df[~agent_mask].copy()
        
        # Spam titles with repeated exclamation
        spam_mask = df['title'].str.contains(r'ด่วน!!!', regex=True, na=False)
        df = df[~spam_mask].copy()
        
        # === GARBAGE PROPERTY TYPES ===
        valid_types = r'คอนโด|condo|condominium|บ้าน|house|detached|ทาวน์|town|ที่ดิน|land'
        garbage_type = ~df['property_type'].astype(str).str.contains(valid_types, case=False, na=False)
        df = df[~garbage_type].copy()
        
        # === NUMERIC FIXES ===
        # Convert price_raw to numeric for checks (keep original)
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        
        # Remove price = 0 or price = 1 (test data)
        zero_price = (temp_price == 0) | (temp_price == 1)
        df = df[~zero_price | temp_price.isna() | df['price_raw'].apply(lambda x: isinstance(x, str) and not x.strip().isdigit())].copy()
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        
        # Fix negative prices/areas — take absolute
        neg_price = temp_price < 0
        df.loc[neg_price, 'price_raw'] = temp_price[neg_price].abs()
        
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        neg_area = temp_area < 0
        df.loc[neg_area, 'area_raw'] = temp_area[neg_area].abs()
        
        # Fix swapped price/area (price < 1000 and area > 100000)
        temp_price = pd.to_numeric(df['price_raw'], errors='coerce')
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        swap_mask = (temp_price < 1000) & (temp_area > 100000) & temp_price.notna() & temp_area.notna()
        if swap_mask.sum() > 0:
            df.loc[swap_mask, ['price_raw', 'area_raw']] = df.loc[swap_mask, ['area_raw', 'price_raw']].values
        
        # Remove bedrooms > 15
        df = df[(df['bedrooms'] <= 15) | df['bedrooms'].isna()].copy()
        
        # Remove area = 999999 (placeholder)
        temp_area = pd.to_numeric(df['area_raw'], errors='coerce')
        df = df[(temp_area != 999999) | temp_area.isna()].copy()
        
        # === DETECT AND FIX HipFlat USD values that are actually THB ===
        # If cost_usd > 500000 it's likely THB not USD
        hipflat_mask = df['source'] == 'HipFlat'
        temp_price_hf = pd.to_numeric(df.loc[hipflat_mask, 'price_raw'], errors='coerce')
        already_thb = temp_price_hf > 500000
        # Mark these so transformer knows not to multiply by 35
        df.loc[hipflat_mask & (temp_price_hf > 500000).reindex(df.index, fill_value=False), 'area_unit'] = 'sqm_split_thb'
        
        self._log_op('spam_outliers', initial_len - len(df))
        return df
        
    def clean(self):
        print("Starting data cleaning...")
        df = self.load_and_merge()
        print(f"\n  Phase 1: Deduplication")
        df = self.deduplicate(df)
        print(f"  Phase 2: Missing Value Handling")
        df = self.handle_missing(df)
        print(f"  Phase 3: Spam & Outlier Filtering")
        df = self.filter_spam_outliers(df)
        
        self.stats['after'] = len(df)
        os.makedirs(self.clean_dir, exist_ok=True)
        df.to_csv(os.path.join(self.clean_dir, 'cleaned_listings.csv'), index=False)
        
        print(f"\n  === Cleaning Summary ===")
        print(f"  Before: {self.stats['before']:,} records")
        print(f"  After:  {self.stats['after']:,} records")
        print(f"  Total removed: {self.stats['before'] - self.stats['after']:,}")
        for op, count in self.stats['operations'].items():
            print(f"    - {op}: {count:,}")
        
        return self.stats

### 2.1 รวมข้อมูลเป็น schema เดียว

`load_and_merge()` อ่านทั้งสาม CSV แล้ว map ชื่อคอลัมน์เป็น `id`, `title`, `description`, `price_raw`, `area_raw`, `area_unit`, `bedrooms`, `bathrooms`, `property_type`, `location`, `floor`, `near_bts`, `posted_date`, `source` และคอลัมน์ที่ดินของ HipFlat

HipFlat ใช้รายชื่อทำเลที่กำหนดไว้ค้นใน title หากไม่พบจะได้ NaN; `near_bts` ของ HipFlat ถูกตั้งเป็น NaN ทุกแถว แล้วรวมด้วย `pd.concat(ignore_index=True)` ส่วน `area_unit` เป็นตัวบอกทั้งหน่วยพื้นที่และเงื่อนไขพิเศษของ HipFlat

In [5]:
cleaning_started = time.perf_counter()
cleaner = DataCleaner(str(RUN_DIR/'data/raw'), str(RUN_DIR/'data/cleaned'))
df = cleaner.load_and_merge()
record_stage('merge', len(df), len(df))
display(df.head())
display(profile(df))
display(df.groupby(['source','area_unit'], dropna=False).size().rename('rows').reset_index())

  Loaded 17200 DDProperty + 17150 Baania + 16000 HipFlat = 50350 total


,id,title,description,price_raw,area_raw,area_unit,bedrooms,bathrooms,property_type,location,floor,near_bts,posted_date,source,land_rai,land_ngan,land_wah
0,DD0,ที่ดิน for sale at Lat Phrao,Beautiful ที่ดิน located in Lat Phrao. Great view.,8459608.0,1.738032e+02,sqm,0,0,ที่ดิน,Lat Phrao,0,NaN,2025-10-25,DDProperty,NaN,NaN,NaN
1,DD1,Townhome for sale at Sathorn,Beautiful Townhome located in Sathorn. Great view.,142.668952,1.860367e+07,sqm,2,2,Townhome,Sathorn,2,BTS สุขุมวิท,2025-12-23,DDProperty,NaN,NaN,NaN
2,DD2,ที่ดิน for sale at หนองจอก,Beautiful ที่ดิน located in หนองจอก. Great view.,33876979.0,2.126623e+03,sqm,0,0,ที่ดิน,หนองจอก,0,NaN,2026-07-17,DDProperty,NaN,NaN,NaN
3,DD3,Single House for sale at Thonburi,Beautiful Single House located in Thonburi. Great view.,11111433.0,1.621918e+02,sqm,2,2,Single House,Thonburi,1,MRT ห้วยขวาง,2026-04-13,DDProperty,NaN,NaN,NaN
4,DD4,land plot for sale at สีลม,Beautiful land plot located in สีลม. Great view.,187062807.0,3.662172e+03,sqm,0,0,land plot,สีลม,0,MRT ลาดพร้าว,2025-11-19,DDProperty,NaN,NaN,NaN


,dtype,missing,missing_pct
id,object,0,0.00
title,object,0,0.00
description,object,0,0.00
price_raw,object,1317,2.62
area_raw,float64,839,1.67
area_unit,object,0,0.00
bedrooms,object,2595,5.15
bathrooms,int64,0,0.00
property_type,object,159,0.32
location,object,0,0.00


,source,area_unit,rows
0,Baania,sqw,17150
1,DDProperty,sqm,17200
2,HipFlat,sqm_split,16000


### 2.2 ลบข้อมูลซ้ำ

ลบซ้ำตาม `id` ก่อน จากนั้นลบตามชุด `title + location + price_raw` โดยเก็บแถวแรก วิธีนี้เป็นการเทียบค่าตรงกันของคอลัมน์ ไม่ได้ใช้ fuzzy similarity หรือเทียบราคาใกล้เคียง แม้ comment/README จะเรียกว่า fuzzy deduplication และทำก่อน normalize ข้อความ/ราคา จึงยังพลาดแถวซ้ำที่เขียนต่างรูปแบบได้

In [6]:
before = len(df)
df = cleaner.deduplicate(df)
record_stage('deduplication', before, len(df))
display(pd.DataFrame(audit))

  [deduplication] Removed 2021 records


,step,before,after,removed
0,merge,50350,50350,0
1,deduplication,50350,48329,2021


### 2.3 จัดการค่าว่างและจำนวนห้องนอน

- ลบแถวที่ราคาและพื้นที่หายพร้อมกัน
- ตั้ง `price_imputed` จากราคาดิบที่เป็น NaN และลบราคา `ราคาต่อรอง` / `สอบถาม`
- แปลง `studio→0`, `3+1→4`, `N/A→NaN` แล้วใช้ `pd.to_numeric(errors='coerce')`
- ห้องนอนที่หายเติมตามข้อความประเภท: คอนโด 1, บ้าน 3, ทาวน์โฮม 2, ที่ดิน 0

ขั้นนี้ยังไม่ได้เติมราคาที่หาย การเติมราคาจะทำหลังแปลงราคาและจัดโซน

In [7]:
before = len(df)
df = cleaner.handle_missing(df)
record_stage('missing_values', before, len(df))
display(profile(df)[['missing','missing_pct']])
display(df['price_imputed'].value_counts(dropna=False))

  [missing_values] Removed 1322 records


,missing,missing_pct
id,0,0.00
title,0,0.00
description,0,0.00
price_raw,310,0.66
area_raw,0,0.00
area_unit,0,0.00
bedrooms,0,0.00
bathrooms,0,0.00
property_type,159,0.34
location,0,0.00


,count
price_imputed,
False,46697
True,310


### 2.4 กรองสแปมและแก้ค่าผิดปกติ

| กฎในโค้ด | วิธีจัดการ |
|---|---|
| artifacts ใน title | ลบ `&amp;`, `<br>`, literal `\n\t` และ emoji ที่ระบุ |
| เบอร์โทรใน title / คำโฆษณานายหน้าใน description / `ด่วน!!!` | ลบแถวด้วย regex |
| ประเภทไม่ตรงคำที่รองรับ | ลบแถว |
| ราคา 0 หรือ 1 | กรองตาม numeric conversion และเงื่อนไขชนิดข้อความในโค้ด |
| ราคา/พื้นที่ติดลบที่แปลงตัวเลขได้ | ใช้ค่าสัมบูรณ์ |
| ราคา <1,000 และพื้นที่ >100,000 | สลับ price_raw กับ area_raw |
| bedrooms >15 / area_raw =999999 | ลบแถว |
| HipFlat price_raw >500,000 | ตั้ง `area_unit=sqm_split_thb` เพื่อไม่คูณ USD อีก |

กฎระบุว่า HipFlat เป็น THB จากระดับราคาเป็น heuristic ของ mock data ไม่ใช่การตรวจ currency metadata; สถิติ `spam_outliers` รวมจำนวนแถวที่ถูกลบ ไม่ได้นับทุก cell ที่แก้ค่า

In [8]:
before = len(df)
df = cleaner.filter_spam_outliers(df)
record_stage('spam_outliers', before, len(df))
cleaner.stats['after'] = len(df)
df.to_csv(RUN_DIR/'data/cleaned/cleaned_listings.csv', index=False)
report['cleaning'] = cleaner.stats
report['cleaning_time'] = time.perf_counter() - cleaning_started
display(pd.DataFrame(audit))
display(df.head())

  [spam_outliers] Removed 3979 records


,step,before,after,removed
0,merge,50350,50350,0
1,deduplication,50350,48329,2021
2,missing_values,48329,47007,1322
3,spam_outliers,47007,43028,3979


,id,title,description,price_raw,area_raw,area_unit,bedrooms,bathrooms,property_type,location,floor,near_bts,posted_date,source,land_rai,land_ngan,land_wah,price_imputed
0,DD0,ที่ดิน for sale at Lat Phrao,Beautiful ที่ดิน located in Lat Phrao. Great view.,8459608.0,173.803238,sqm,0.0,0,ที่ดิน,Lat Phrao,0,NaN,2025-10-25,DDProperty,NaN,NaN,NaN,False
1,DD1,Townhome for sale at Sathorn,Beautiful Townhome located in Sathorn. Great view.,18603671.0,142.668952,sqm,2.0,2,Townhome,Sathorn,2,BTS สุขุมวิท,2025-12-23,DDProperty,NaN,NaN,NaN,False
2,DD2,ที่ดิน for sale at หนองจอก,Beautiful ที่ดิน located in หนองจอก. Great view.,33876979.0,2126.623114,sqm,0.0,0,ที่ดิน,หนองจอก,0,NaN,2026-07-17,DDProperty,NaN,NaN,NaN,False
3,DD3,Single House for sale at Thonburi,Beautiful Single House located in Thonburi. Great view.,11111433.0,162.191800,sqm,2.0,2,Single House,Thonburi,1,MRT ห้วยขวาง,2026-04-13,DDProperty,NaN,NaN,NaN,False
4,DD4,land plot for sale at สีลม,Beautiful land plot located in สีลม. Great view.,187062807.0,3662.172344,sqm,0.0,0,land plot,สีลม,0,MRT ลาดพร้าว,2025-11-19,DDProperty,NaN,NaN,NaN,False


## 3. แปลงหน่วยและสร้าง features

นิยาม `DataTransformer` คงโค้ดต้นฉบับ รวมทั้งกฎกรองและค่าสมมติฐาน การเรียกด้านล่างเริ่มจากอ่าน cleaned CSV ใหม่เช่นเดียวกับ `transform()`

แหล่งโค้ด: [src/transformation/transformer.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/transformation/transformer.py)

In [9]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

class DataTransformer:
    """Handles feature engineering and standardization."""
    
    def __init__(self, clean_dir, trans_dir):
        self.clean_dir = clean_dir
        self.trans_dir = trans_dir
        
    def normalize_prices(self, df):
        """Parse text prices and convert USD to THB."""
        def parse_price(val, source, area_unit):
            if pd.isna(val):
                return np.nan
            
            # Numeric values
            if isinstance(val, (int, float)):
                num = float(val)
                if num <= 0:
                    return np.nan
                # HipFlat: convert USD to THB unless already marked as THB
                if source == 'HipFlat':
                    if area_unit == 'sqm_split_thb':
                        return num  # Already THB
                    return num * 35.0
                return num
                
            val_str = str(val).strip()
            
            # Thai formats: "3.5 ล้าน", "3.5ล้าน", "35 ล้านบาท"
            lan_match = re.search(r'([\d,.]+)\s*ล้าน', val_str)
            if lan_match:
                num = float(lan_match.group(1).replace(',', ''))
                return num * 1_000_000
            
            # Thai: "8 แสน"
            saen_match = re.search(r'([\d,.]+)\s*แสน', val_str)
            if saen_match:
                num = float(saen_match.group(1).replace(',', ''))
                return num * 100_000
                
            # Numeric with commas and optional "บาท": "3,500,000 บาท", "3500000"
            val_str = val_str.replace(',', '').replace('บาท', '').replace(' ', '').strip()
            try:
                num = float(val_str)
                if num <= 0:
                    return np.nan
                if source == 'HipFlat' and area_unit != 'sqm_split_thb':
                    return num * 35.0
                return num
            except:
                return np.nan
        
        import re
        df['price_thb'] = df.apply(
            lambda row: parse_price(row['price_raw'], row['source'], row.get('area_unit', '')), 
            axis=1
        )
        
        # Remove extreme prices (> 500M THB for any single listing is unrealistic for mock data)
        df.loc[df['price_thb'] > 500_000_000, 'price_thb'] = np.nan
        
        return df

    def convert_area(self, df):
        """Convert all areas to SQM."""
        def to_sqm(row):
            area = pd.to_numeric(row.get('area_raw'), errors='coerce')
            unit = row.get('area_unit', 'sqm')
            
            # For HipFlat with split rai/ngan/wah
            if unit in ['sqm_split', 'sqm_split_thb']:
                rai = pd.to_numeric(row.get('land_rai'), errors='coerce')
                ngan = pd.to_numeric(row.get('land_ngan'), errors='coerce')
                wah = pd.to_numeric(row.get('land_wah'), errors='coerce')
                
                if pd.notna(rai) and pd.notna(ngan) and pd.notna(wah):
                    # Cap rai to reasonable values (max ~10 rai for Bangkok)
                    rai = min(rai, 10)
                    ngan = min(ngan, 3)
                    sqm = (rai * 1600) + (ngan * 400) + (wah * 4)
                    if sqm > 0:
                        return sqm
                
                # Fallback to area column
                if pd.notna(area) and area > 0:
                    return area
                return np.nan
            
            if pd.isna(area) or area <= 0:
                return np.nan
                
            if unit == 'sqw':
                return area * 4.0  # ตร.วา to ตร.ม.
            
            return area  # Already in sqm
            
        df['area_sqm'] = df.apply(to_sqm, axis=1)
        
        # Cap extreme areas (max ~20,000 sqm for urban plots)
        df.loc[df['area_sqm'] > 20000, 'area_sqm'] = np.nan
        
        return df

    def categorize_location(self, df):
        """Normalize locations and assign zones."""
        loc_map = {
            'sukhumvit': 'สุขุมวิท', 'สุขุมวิท ': 'สุขุมวิท', ' สุขุมวิท': 'สุขุมวิท', 'สุขุมวิทร์': 'สุขุมวิท',
            'silom': 'สีลม', 'sathorn': 'สาทร', 'lat phrao': 'ลาดพร้าว', 'rama 9': 'พระราม9',
            'on nut': 'อ่อนนุช', 'bang na': 'บางนา', 'rangsit': 'รังสิต', 'thonburi': 'ธนบุรี',
            'phaya thai': 'พญาไท', 'ari': 'อารีย์', 'thong lo': 'ทองหล่อ', 'ekkamai': 'เอกมัย',
            'ratchathewi': 'ราชเทวี', 'huai khwang': 'ห้วยขวาง', 'bang kapi': 'บางกะปิ',
            'min buri': 'มีนบุรี', 'nong chok': 'หนองจอก', 'taling chan': 'ตลิ่งชัน',
            'bang khae': 'บางแค', 'chatuchak': 'จตุจักร'
        }
        
        def normalize_loc(loc):
            if pd.isna(loc):
                return np.nan
            loc_str = str(loc).strip().lower()
            return loc_map.get(loc_str, loc.strip() if isinstance(loc, str) else loc)
        
        df['location'] = df['location'].apply(normalize_loc)
        
        zones = {
            'CBD': ['สุขุมวิท', 'สีลม', 'สาทร', 'ทองหล่อ', 'เอกมัย'],
            'Inner City': ['ลาดพร้าว', 'พระราม9', 'อ่อนนุช', 'พญาไท', 'อารีย์', 'ราชเทวี', 'ห้วยขวาง', 'จตุจักร'],
            'Suburban': ['บางนา', 'บางกะปิ', 'ธนบุรี', 'ตลิ่งชัน', 'บางแค'],
            'Outer': ['รังสิต', 'มีนบุรี', 'หนองจอก']
        }
        
        def assign_zone(loc):
            if pd.isna(loc):
                return 'Unknown'
            for z, locs in zones.items():
                if loc in locs:
                    return z
            return 'Unknown'
            
        df['zone'] = df['location'].apply(assign_zone)
        df['district'] = df['location']
        return df

    def feature_engineering(self, df):
        """Create derived features for ML."""
        # Normalize property type
        p_map = {
            'condo': 'คอนโด', 'condominium': 'คอนโด', 
            'house': 'บ้านเดี่ยว', 'single house': 'บ้านเดี่ยว', 'detached house': 'บ้านเดี่ยว',
            'townhome': 'ทาวน์โฮม', 'townhouse': 'ทาวน์โฮม',
            'land': 'ที่ดิน', 'land plot': 'ที่ดิน'
        }
        df['property_type'] = df['property_type'].apply(
            lambda x: p_map.get(str(x).lower().strip(), x) if pd.notna(x) else x
        )
        
        # Impute missing prices using median by (property_type, zone)
        for (ptype, zone), group in df.groupby(['property_type', 'zone']):
            median_price = group['price_thb'].median()
            if pd.notna(median_price):
                mask = (df['property_type'] == ptype) & (df['zone'] == zone) & df['price_thb'].isna()
                df.loc[mask, 'price_thb'] = median_price
                df.loc[mask, 'price_imputed'] = True
        
        # Fill remaining NaN prices with overall median
        overall_median = df['price_thb'].median()
        df['price_thb'] = df['price_thb'].fillna(overall_median)
        
        # Price per sqm
        df['price_per_sqm'] = np.where(
            (df['area_sqm'] > 0) & df['area_sqm'].notna(),
            df['price_thb'] / df['area_sqm'],
            np.nan
        )
        
        # BTS
        df['has_bts'] = df['near_bts'].apply(lambda x: 1 if pd.notna(x) and str(x).strip() != '' else 0)
        
        # Age
        ref_date = datetime.now()
        df['posted_date_parsed'] = pd.to_datetime(df['posted_date'], errors='coerce', dayfirst=True)
        # Try alternative formats for unparsed dates
        unparsed = df['posted_date_parsed'].isna() & df['posted_date'].notna()
        if unparsed.any():
            df.loc[unparsed, 'posted_date_parsed'] = pd.to_datetime(
                df.loc[unparsed, 'posted_date'], errors='coerce', format='mixed'
            )
        df['age_days'] = (ref_date - df['posted_date_parsed']).dt.days.fillna(180)
        
        # Floor handling
        df['floor'] = pd.to_numeric(df['floor'], errors='coerce').fillna(0)
        
        return df

    def transform(self):
        print("Starting data transformation...")
        df = pd.read_csv(os.path.join(self.clean_dir, 'cleaned_listings.csv'), low_memory=False)
        
        print("  Step 1: Price normalization...")
        df = self.normalize_prices(df)
        print("  Step 2: Area unit conversion...")
        df = self.convert_area(df)
        print("  Step 3: Location categorization...")
        df = self.categorize_location(df)
        print("  Step 4: Feature engineering...")
        df = self.feature_engineering(df)
        
        # Drop records where both price and area are still missing
        df = df.dropna(subset=['price_thb'])
        df = df[df['area_sqm'].notna() & (df['area_sqm'] > 0)]
        
        # Remove extreme price_per_sqm outliers
        df = df[df['price_per_sqm'].notna() & (df['price_per_sqm'] > 0)]
        q99 = df['price_per_sqm'].quantile(0.99)
        df = df[df['price_per_sqm'] <= q99]
        
        cols = ['id', 'title', 'price_thb', 'area_sqm', 'price_per_sqm', 'bedrooms', 'bathrooms', 
                'property_type', 'location', 'zone', 'district', 'floor', 'near_bts', 'has_bts', 
                'posted_date', 'age_days', 'source', 'price_imputed']
        
        # Ensure cols exist
        for c in cols:
            if c not in df.columns:
                df[c] = np.nan
                
        final_df = df[cols].copy()
        
        # Fill NaN bedrooms/bathrooms
        final_df['bedrooms'] = final_df['bedrooms'].fillna(1)
        final_df['bathrooms'] = final_df['bathrooms'].fillna(1)
        
        os.makedirs(self.trans_dir, exist_ok=True)
        final_df.to_csv(os.path.join(self.trans_dir, 'final_listings.csv'), index=False)
        
        print(f"\n  === Transformation Summary ===")
        print(f"  Final records: {len(final_df):,}")
        print(f"  By zone: {final_df['zone'].value_counts().to_dict()}")
        print(f"  By type: {final_df['property_type'].value_counts().to_dict()}")
        print(f"  Price range: {final_df['price_thb'].min():,.0f} — {final_df['price_thb'].max():,.0f} THB")
        print(f"  Area range: {final_df['area_sqm'].min():,.1f} — {final_df['area_sqm'].max():,.1f} sqm")
        
        return len(final_df)

In [10]:
transformation_started = time.perf_counter()
transformer = DataTransformer(str(RUN_DIR/'data/cleaned'), str(RUN_DIR/'data/transformed'))
df = pd.read_csv(RUN_DIR/'data/cleaned/cleaned_listings.csv', low_memory=False)

### 3.1 แปลงราคาเป็น THB

ใช้ regex แปลง `3.5 ล้าน→3,500,000`, `8 แสน→800,000` และลบ comma/คำว่าบาท ราคา HipFlat คูณ 35 ยกเว้นแถวที่ cleaner ระบุเป็น THB; 35 เป็นค่าคงที่ในโปรเจกต์ ไม่ใช่อัตราแลกเปลี่ยนล่าสุด ราคาที่แปลงไม่ได้/ไม่เป็นบวกเป็น NaN และราคามากกว่า 500 ล้านบาทตั้งเป็น NaN เพื่อเติมภายหลัง

In [11]:
df = transformer.normalize_prices(df)
display(df[['source','price_raw','area_unit','price_thb']].sample(8, random_state=42))
print('Missing normalized price:', int(df['price_thb'].isna().sum()))

Missing normalized price: 335


,source,price_raw,area_unit,price_thb
36196,HipFlat,9063632.79618051,sqm_split_thb,9.063633e+06
38843,HipFlat,2120037.587977841,sqm_split_thb,2.120038e+06
21410,Baania,22189019,sqw,2.218902e+07
25141,Baania,109618846,sqw,1.096188e+08
39506,HipFlat,920919.1909888352,sqm_split_thb,9.209192e+05
7567,DDProperty,9687137.0,sqm,9.687137e+06
30144,HipFlat,97406.04964903196,sqm_split,3.409212e+06
5756,DDProperty,119007028.0,sqm,1.190070e+08


### 3.2 แปลงพื้นที่เป็นตารางเมตร

ตร.วา ×4; ไร่ ×1,600 + งาน ×400 + วา ×4 สำหรับ HipFlat หากส่วนไร่/งาน/วามีครบ ให้คำนวณจากส่วนเหล่านั้นก่อนและ cap ไร่ไม่เกิน 10 งานไม่เกิน 3; ถ้าใช้ไม่ได้จึง fallback ไป `area_raw` พื้นที่ไม่เป็นบวกหรือมากกว่า 20,000 ตร.ม. เป็น NaN

In [12]:
df = transformer.convert_area(df)
display(df[['source','area_raw','area_unit','land_rai','land_ngan','land_wah','area_sqm']].sample(8, random_state=42))
print('Missing normalized area:', int(df['area_sqm'].isna().sum()))

Missing normalized area: 73


,source,area_raw,area_unit,land_rai,land_ngan,land_wah,area_sqm
36196,HipFlat,4104.261341,sqm_split_thb,2.0,2.0,26.065335,4104.261341
38843,HipFlat,3507.311553,sqm_split_thb,2.0,0.0,76.827888,3507.311553
21410,Baania,92.086390,sqw,NaN,NaN,NaN,368.345561
25141,Baania,624.944753,sqw,NaN,NaN,NaN,2499.779011
39506,HipFlat,354.669937,sqm_split_thb,0.0,0.0,88.667484,354.669937
7567,DDProperty,125.577420,sqm,NaN,NaN,NaN,125.577420
30144,HipFlat,95.576270,sqm_split,0.0,0.0,23.894067,95.576270
5756,DDProperty,1169.488413,sqm,NaN,NaN,NaN,1169.488413


### 3.3 Normalize ทำเลและจัดโซน

ตัดช่องว่าง/แปลงเป็น lowercase เพื่อ lookup ทำเล เช่น Sukhumvit→สุขุมวิท และแก้บางรูปสะกด จากนั้นจัดโซนตามรายชื่อที่กำหนดใน source; ทำเลที่ไม่พบได้ Unknown ส่วน `district` เป็นสำเนาของ location ไม่ใช่เขตการปกครองที่ตรวจสอบแล้ว

| โซน | ทำเล |
|---|---|
| CBD | สุขุมวิท สีลม สาทร ทองหล่อ เอกมัย |
| Inner City | ลาดพร้าว พระราม9 อ่อนนุช พญาไท อารีย์ ราชเทวี ห้วยขวาง จตุจักร |
| Suburban | บางนา บางกะปิ ธนบุรี ตลิ่งชัน บางแค |
| Outer | รังสิต มีนบุรี หนองจอก |

In [13]:
df = transformer.categorize_location(df)
display(df[['location','zone','district']].drop_duplicates().sort_values('zone'))
display(df['zone'].value_counts(dropna=False))

,location,zone,district
12,สุขุมวิท,CBD,สุขุมวิท
6,เอกมัย,CBD,เอกมัย
4,สีลม,CBD,สีลม
90,ทองหล่อ,CBD,ทองหล่อ
1,สาทร,CBD,สาทร
19,ราชเทวี,Inner City,ราชเทวี
25,อ่อนนุช,Inner City,อ่อนนุช
24,พระราม9,Inner City,พระราม9
21,ห้วยขวาง,Inner City,ห้วยขวาง
20,อารีย์,Inner City,อารีย์


,count
zone,
Inner City,16432
Suburban,10274
CBD,10176
Outer,6146


### 3.4 สร้าง features และเติมราคา

1. Normalize ชื่อ property_type ภาษาอังกฤษเป็นประเภทภาษาไทย
2. เติม price_thb ด้วย median ของกลุ่ม property_type + zone และตั้ง price_imputed=True ให้แถวที่เติมในกลุ่ม หากเหลือ NaN ใช้ median รวม
3. สร้าง price_per_sqm = price_thb / area_sqm เมื่อพื้นที่เป็นบวก
4. has_bts=1 เมื่อ near_bts ไม่ว่าง มิฉะนั้น 0 จึงรวม MRT ด้วยและ HipFlat ที่ไม่มีข้อมูลจะถูกนับเป็น 0
5. Parse posted_date ด้วย dayfirst=True และ fallback format='mixed'; age_days เป็นจำนวนวันจาก datetime.now() ถึงวันประกาศ ถ้า parse ไม่ได้ใช้ 180
6. floor แปลงตัวเลขและเติม 0

flag price_imputed ของ fallback median รวมไม่ได้ถูกตั้งเพิ่มโดยตรงในต้นฉบับ จึงควรตรวจหากใช้กับข้อมูลชุดอื่น

In [14]:
price_missing_before = df['price_thb'].isna().sum()
df = transformer.feature_engineering(df)
print('Missing price before/after:', int(price_missing_before), int(df['price_thb'].isna().sum()))
display(df[['property_type','zone','price_thb','area_sqm','price_per_sqm','has_bts','age_days','price_imputed']].head(8))

<string>:184: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
Missing price before/after: 335 0


,property_type,zone,price_thb,area_sqm,price_per_sqm,has_bts,age_days,price_imputed
0,ที่ดิน,Inner City,8459608.0,173.803238,48673.477435,0,346,False
1,ทาวน์โฮม,CBD,18603671.0,142.668952,130397.474522,1,287,False
2,ที่ดิน,Outer,33876979.0,2126.623114,15929.940186,0,81,False
3,บ้านเดี่ยว,Suburban,11111433.0,162.191800,68507.982556,1,176,False
4,ที่ดิน,CBD,187062807.0,3662.172344,51079.738860,1,321,False
5,บ้านเดี่ยว,Suburban,9905907.0,176.915436,55992.327285,1,40,False
6,คอนโด,CBD,15252586.0,73.645381,207108.523132,1,311,False
7,ทาวน์โฮม,Outer,3184742.0,82.876349,38427.633861,0,112,False


### 3.5 กรองข้อมูลสุดท้ายและบันทึก

ลบราคา NaN แล้วเก็บพื้นที่ที่ไม่ว่างและ >0; เก็บ price_per_sqm ที่ไม่ว่างและ >0 จากนั้นตัดค่าที่สูงกว่า percentile 99 ของข้อมูลที่เหลือ **โค้ดนี้ไม่ได้ใช้ IQR ตามที่ README ระบุ** เลือก 18 คอลัมน์สุดท้ายแล้วเติม bedrooms/bathrooms ที่ยังหายด้วย 1 โค้ดด้านล่างถอดจากส่วนท้าย `transform()` เพื่อให้ตรวจผลแต่ละช่วงได้

In [15]:
before = len(df)
df = df.dropna(subset=['price_thb'])
df = df[df['area_sqm'].notna() & (df['area_sqm'] > 0)]
record_stage('required_price_area', before, len(df))
before = len(df)
df = df[df['price_per_sqm'].notna() & (df['price_per_sqm'] > 0)]
record_stage('positive_price_per_sqm', before, len(df))
q99 = df['price_per_sqm'].quantile(0.99)
before = len(df)
df = df[df['price_per_sqm'] <= q99]
record_stage('price_per_sqm_p99', before, len(df))
cols = ['id','title','price_thb','area_sqm','price_per_sqm','bedrooms','bathrooms',
        'property_type','location','zone','district','floor','near_bts','has_bts',
        'posted_date','age_days','source','price_imputed']
for c in cols:
    if c not in df.columns:
        df[c] = np.nan
final_df = df[cols].copy()
final_df['bedrooms'] = final_df['bedrooms'].fillna(1)
final_df['bathrooms'] = final_df['bathrooms'].fillna(1)
final_df.to_csv(RUN_DIR/'data/transformed/final_listings.csv', index=False)
report['transformation'] = {'final_record_count': len(final_df)}
report['transformation_time'] = time.perf_counter() - transformation_started
print('P99 threshold:', q99)
display(pd.DataFrame(audit))
display(final_df.head())
display(profile(final_df))

P99 threshold: 274654.9917818666


,step,before,after,removed
0,merge,50350,50350,0
1,deduplication,50350,48329,2021
2,missing_values,48329,47007,1322
3,spam_outliers,47007,43028,3979
4,required_price_area,43028,42955,73
5,positive_price_per_sqm,42955,42955,0
6,price_per_sqm_p99,42955,42525,430


,id,title,price_thb,area_sqm,price_per_sqm,bedrooms,bathrooms,property_type,location,zone,district,floor,near_bts,has_bts,posted_date,age_days,source,price_imputed
0,DD0,ที่ดิน for sale at Lat Phrao,8459608.0,173.803238,48673.477435,0.0,0,ที่ดิน,ลาดพร้าว,Inner City,ลาดพร้าว,0,NaN,0,2025-10-25,346,DDProperty,False
1,DD1,Townhome for sale at Sathorn,18603671.0,142.668952,130397.474522,2.0,2,ทาวน์โฮม,สาทร,CBD,สาทร,2,BTS สุขุมวิท,1,2025-12-23,287,DDProperty,False
2,DD2,ที่ดิน for sale at หนองจอก,33876979.0,2126.623114,15929.940186,0.0,0,ที่ดิน,หนองจอก,Outer,หนองจอก,0,NaN,0,2026-07-17,81,DDProperty,False
3,DD3,Single House for sale at Thonburi,11111433.0,162.191800,68507.982556,2.0,2,บ้านเดี่ยว,ธนบุรี,Suburban,ธนบุรี,1,MRT ห้วยขวาง,1,2026-04-13,176,DDProperty,False
4,DD4,land plot for sale at สีลม,187062807.0,3662.172344,51079.738860,0.0,0,ที่ดิน,สีลม,CBD,สีลม,0,MRT ลาดพร้าว,1,2025-11-19,321,DDProperty,False


,dtype,missing,missing_pct
id,object,0,0.00
title,object,0,0.00
price_thb,float64,0,0.00
area_sqm,float64,0,0.00
price_per_sqm,float64,0,0.00
bedrooms,float64,0,0.00
bathrooms,int64,0,0.00
property_type,object,0,0.00
location,object,0,0.00
zone,object,0,0.00


### 3.6 ตรวจคุณภาพหลังแปลง (เพิ่มสำหรับ notebook)

ตรวจเงื่อนไขที่ source ตั้งใจทำ และดูสิ่งที่ยังเหลือโดยไม่แก้ข้อมูลเพิ่ม เช่น duplicate ที่หลุดเพราะ normalize หลัง deduplication และ date ที่ parse ไม่ได้

In [16]:
checks = pd.Series({
    'unique_id': final_df['id'].is_unique,
    'price_present_and_positive': bool((final_df['price_thb'].notna() & (final_df['price_thb'] > 0)).all()),
    'area_present_and_positive': bool((final_df['area_sqm'].notna() & (final_df['area_sqm'] > 0)).all()),
    'price_per_sqm_consistent': bool(np.allclose(final_df['price_per_sqm'], final_df['price_thb']/final_df['area_sqm'])),
    'has_bts_binary': bool(final_df['has_bts'].isin([0,1]).all()),
    'exactly_18_columns': len(final_df.columns)==18,
}, name='passed')
display(checks.to_frame())
print('Duplicates after normalization (type, location, price, area):', int(final_df.duplicated(['property_type','location','price_thb','area_sqm']).sum()))
print('Unparsed dates before final column selection:', int(df['posted_date_parsed'].isna().sum()))
assert checks.all(), 'Review failed data quality checks'

Duplicates after normalization (type, location, price, area): 0
Unparsed dates before final column selection: 0


,passed
unique_id,True
price_present_and_positive,True
area_present_and_positive,True
price_per_sqm_consistent,True
has_bts_binary,True
exactly_18_columns,True


## 4. เตรียมข้อมูลเข้าโมเดล

ส่วนนี้เก็บนิยาม predictor จาก repo: ตั้ง floor ของที่ดินเป็น 0, ลบแถวที่ price/area หาย, เติม numeric ที่กำหนด แล้วเลือก 8 features: area_sqm, bedrooms, bathrooms, floor, zone, property_type, has_bts, age_days ใช้ get_dummies กับ zone/property_type แล้วแบ่ง train/test 80:20 random_state=42 เป้าหมายคือ price_thb; ไม่ใช้ price_per_sqm เป็น input

ไม่มี scaling ใน source และ One-hot encoding ทำก่อน split ข้อมูล

แหล่งโค้ด: [src/modeling/predictor.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/modeling/predictor.py)

In [17]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import json

class PropertyPredictor:
    """Trains models and makes predictions for property prices."""
    
    def __init__(self, data_path, model_dir):
        self.data_path = data_path
        self.model_dir = model_dir
        self.best_model = None
        self.model_columns = None
        
    def prepare_data(self):
        df = pd.read_csv(self.data_path)
        
        # Fill missing numeric values
        df['floor'] = df.apply(lambda r: 0 if r['property_type'] == 'ที่ดิน' else r['floor'], axis=1)
        df = df.dropna(subset=['price_thb', 'area_sqm']) # Essential cols
        df = df.fillna({'bedrooms': 1, 'bathrooms': 1, 'floor': 1, 'age_days': 0})
        
        # One-hot encoding
        features = df[['area_sqm', 'bedrooms', 'bathrooms', 'floor', 'zone', 'property_type', 'has_bts', 'age_days']]
        features = pd.get_dummies(features, columns=['zone', 'property_type'])
        
        X = features
        y = df['price_thb']
        self.model_columns = list(X.columns)
        
        return train_test_split(X, y, test_size=0.2, random_state=42)

    def train_and_evaluate(self):
        print("Starting model training...")
        X_train, X_test, y_train, y_test = self.prepare_data()
        
        models = {
            'Linear Regression': LinearRegression(),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, random_state=42)
        }
        
        results = {}
        best_r2 = -float('inf')
        best_name = ""
        
        for name, model in models.items():
            print(f"Training {name}...")
            model.fit(X_train, y_train)
            preds = model.predict(X_test)
            
            mae = mean_absolute_error(y_test, preds)
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            r2 = r2_score(y_test, preds)
            
            results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
            
            if r2 > best_r2:
                best_r2 = r2
                self.best_model = model
                best_name = name
                
        print(f"Best model: {best_name} (R2: {best_r2:.4f})")
        
        # Save artifacts
        os.makedirs(self.model_dir, exist_ok=True)
        joblib.dump(self.best_model, os.path.join(self.model_dir, 'best_model.pkl'))
        joblib.dump(self.model_columns, os.path.join(self.model_dir, 'model_columns.pkl'))
        
        with open(os.path.join(self.model_dir, 'evaluation.json'), 'w') as f:
            json.dump(results, f, indent=4)
            
        # Feature importance if available
        if hasattr(self.best_model, 'feature_importances_'):
            fi = dict(zip(self.model_columns, self.best_model.feature_importances_))
            with open(os.path.join(self.model_dir, 'feature_importance.json'), 'w') as f:
                json.dump(fi, f, indent=4)
                
        return results

    @classmethod
    def load_model(cls, model_dir):
        inst = cls(None, model_dir)
        inst.best_model = joblib.load(os.path.join(model_dir, 'best_model.pkl'))
        inst.model_columns = joblib.load(os.path.join(model_dir, 'model_columns.pkl'))
        return inst
        
    def predict(self, area_sqm, bedrooms, bathrooms, floor, zone, property_type, has_bts):
        # Create input df
        input_data = pd.DataFrame([{
            'area_sqm': area_sqm, 'bedrooms': bedrooms, 'bathrooms': bathrooms,
            'floor': floor, 'has_bts': has_bts, 'age_days': 0,
            f'zone_{zone}': 1, f'property_type_{property_type}': 1
        }])
        
        # Reindex to match training columns
        input_data = input_data.reindex(columns=self.model_columns, fill_value=0)
        return self.best_model.predict(input_data)[0]

In [18]:
predictor = PropertyPredictor(str(RUN_DIR/'data/transformed/final_listings.csv'), str(RUN_DIR/'models'))
X_train, X_test, y_train, y_test = predictor.prepare_data()
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Features:', predictor.model_columns)
display(X_train.head())

<string>:21: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
Train: (34020, 14) Test: (8505, 14)
Features: ['area_sqm', 'bedrooms', 'bathrooms', 'floor', 'has_bts', 'age_days', 'zone_CBD', 'zone_Inner City', 'zone_Outer', 'zone_Suburban', 'property_type_คอนโด', 'property_type_ทาวน์โฮม', 'property_type_ที่ดิน', 'property_type_บ้านเดี่ยว']


,area_sqm,bedrooms,bathrooms,floor,has_bts,age_days,zone_CBD,zone_Inner City,zone_Outer,zone_Suburban,property_type_คอนโด,property_type_ทาวน์โฮม,property_type_ที่ดิน,property_type_บ้านเดี่ยว
22453,70.607739,2.0,2,3,1,6,True,False,False,False,False,True,False,False
5026,166.316836,3.0,2,3,1,148,False,False,True,False,False,False,False,True
1248,369.614010,7.0,6,1,0,345,True,False,False,False,False,False,False,True
40574,138.567989,3.0,2,19,0,28,False,False,False,True,True,False,False,False
29974,37.217692,1.0,1,28,0,286,False,True,False,False,True,False,False,False


### 4.1 เทรนและประเมิน (ส่วนต่อจากการเตรียมข้อมูล)

ต้นฉบับเทียบ Linear Regression, Random Forest (100 trees) และ Gradient Boosting (200 estimators) ด้วย MAE, RMSE และ R² เลือกโมเดล R² สูงสุดและบันทึกโมเดล, ลำดับคอลัมน์, evaluation และ feature importance ถ้ามี `TRAIN_MODELS=False` จะข้ามส่วนนี้ได้

ข้อจำกัดการประเมิน: ราคาที่เติมถูกใช้เป็น target ด้วย และ median/P99 คำนวณจากข้อมูลทั้งหมดก่อน split จึงมีข้อมูลของชุดทดสอบเข้ามาใน preprocessing ผลนี้เหมาะใช้สาธิต pipeline ข้อมูลจำลอง การประเมินเพื่อใช้งานจริงควรแยกชุดข้อมูลก่อนเรียนรู้สถิติและประเมินด้วยราคาที่สังเกตจริง ข้อเสนอเหล่านี้ไม่ได้เปลี่ยนโค้ดต้นฉบับในไฟล์นี้

In [19]:
if TRAIN_MODELS:
    t0 = time.perf_counter()
    evaluation = predictor.train_and_evaluate()
    report['modeling'] = evaluation
    report['modeling_time'] = time.perf_counter() - t0
    display(pd.DataFrame(evaluation).T.sort_values('R2', ascending=False))
else:
    print('Model training skipped')

Starting model training...
<string>:21: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
Training Linear Regression...
Training Random Forest...
Training Gradient Boosting...
Best model: Gradient Boosting (R2: 0.6957)


,MAE,RMSE,R2
Gradient Boosting,1.588634e+07,3.285828e+07,0.695675
Random Forest,1.528228e+07,3.474768e+07,0.659670
Linear Regression,2.432941e+07,4.136327e+07,0.517744


### 4.2 เตรียม input ใหม่ตามคอลัมน์ที่เทรน

predict() สร้าง one-hot ของโซน/ประเภท จากนั้น reindex ให้คอลัมน์ตรงกับ model_columns เติมคอลัมน์อื่นด้วย 0 และตั้ง age_days=0 เหมือน endpoint ทำนายของ dashboard

In [20]:
if TRAIN_MODELS:
    predicted_price = predictor.predict(area_sqm=50, bedrooms=1, bathrooms=1, floor=10,
                                        zone='CBD', property_type='คอนโด', has_bts=1)
    print(f'Example predicted price on mock data: {predicted_price:,.2f} THB')

Example predicted price on mock data: 14,536,763.96 THB


## 5. ส่งข้อมูลต่อ dashboard

server.py โหลด final_listings.csv และ model artifacts ไปใช้ต่อ ตัวอย่างด้านล่างรวบรวมการคำนวณข้อมูลที่ส่งไปหน้าเว็บโดยไม่เปิด Flask server

งาน dashboard: นับประกาศ, mean/median ราคา, mean ราคา/ตร.ม., groupby ประเภท/โซน/แหล่ง; histogram จำกัดข้อมูลถึง P95 และแบ่ง 20 bins; scatter เลือกพื้นที่ <2,000 ตร.ม. ราคา <P95 และ sample ไม่เกิน 500 แถว; trend parse วันแล้วเฉลี่ยรายเดือน; ตารางกรองตามประเภท โซน ราคาและข้อความ ก่อน sort/pagination

P95 เป็นตัวกรองเพื่อแสดงกราฟ ไม่ใช่ขั้นตอนที่เปลี่ยน final CSV; monthly trend ใน server parse วันที่ต่างจาก transformer จึงอาจอ่านวันที่หลายรูปแบบได้ไม่ครบ

แหล่งโค้ด: [app/server.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/app/server.py)

In [21]:
dashboard_df = pd.read_csv(RUN_DIR/'data/transformed/final_listings.csv')
summary = pd.Series({
    'total_listings': len(dashboard_df),
    'avg_price': dashboard_df['price_thb'].mean(),
    'median_price': dashboard_df['price_thb'].median(),
    'avg_price_per_sqm': dashboard_df['price_per_sqm'].mean()
})
display(summary.to_frame('value'))
for group in ['property_type', 'zone', 'source']:
    display(dashboard_df.groupby(group).agg(count=('price_thb','count'), avg_price=('price_thb','mean')))
prices = dashboard_df['price_thb'].dropna()
cap = prices.quantile(0.95)
hist, bins = np.histogram(prices[prices <= cap], bins=20)
display(pd.DataFrame({'from_thb':bins[:-1], 'to_thb':bins[1:], 'count':hist}))
scatter = dashboard_df[(dashboard_df['area_sqm']>0) & (dashboard_df['area_sqm']<2000)
                       & (dashboard_df['price_thb']>0) & (dashboard_df['price_thb']<cap)]
if len(scatter)>500:
    scatter = scatter.sample(500, random_state=42)
print('Scatter sample rows:', len(scatter))
trend = dashboard_df.copy()
trend['posted_date_dt'] = pd.to_datetime(trend['posted_date'], errors='coerce')
trend = trend.dropna(subset=['posted_date_dt','price_thb'])
display(trend.groupby(trend['posted_date_dt'].dt.to_period('M'))['price_thb'].mean().to_frame('avg_price'))
print('Dates excluded from dashboard trend:', len(dashboard_df)-len(trend))
# Example: condo listings in CBD; price descending; first page of 20 rows.
page = dashboard_df[(dashboard_df['property_type']=='คอนโด') & (dashboard_df['zone']=='CBD')]
display(page.sort_values('price_thb', ascending=False).iloc[:20][['id','title','price_thb','area_sqm','source']])

<string>:1: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
Scatter sample rows: 500
Dates excluded from dashboard trend: 15325


,value
total_listings,4.252500e+04
avg_price,3.205436e+07
median_price,1.199711e+07
avg_price_per_sqm,7.337125e+04


,count,avg_price
property_type,,
คอนโด,9392,1.294092e+07
ทาวน์โฮม,9784,9.252436e+06
ที่ดิน,9992,8.832201e+07
บ้านเดี่ยว,13357,2.010416e+07


,count,avg_price
zone,,
CBD,9752,4.915822e+07
Inner City,16381,3.445128e+07
Outer,6137,1.577777e+07
Suburban,10255,2.170124e+07


,count,avg_price
source,,
Baania,13295,4.606869e+07
DDProperty,14183,4.624398e+07
HipFlat,15047,6.296943e+06


,from_thb,to_thb,count
0,5.000948e+05,8.529738e+06,15958
1,8.529738e+06,1.655938e+07,10548
2,1.655938e+07,2.458902e+07,4435
3,2.458902e+07,3.261867e+07,2464
4,3.261867e+07,4.064831e+07,1662
5,4.064831e+07,4.867795e+07,1215
6,4.867795e+07,5.670759e+07,781
7,5.670759e+07,6.473724e+07,573
8,6.473724e+07,7.276688e+07,402
9,7.276688e+07,8.079652e+07,333


,avg_price
posted_date_dt,
2025-10,3.560001e+07
2025-11,3.760504e+07
2025-12,3.727086e+07
2026-01,3.777882e+07
2026-02,3.873285e+07
2026-03,3.785405e+07
2026-04,3.663089e+07
2026-05,3.948236e+07
2026-06,3.633944e+07


,id,title,price_thb,area_sqm,source
14709,BN689,คอนโด for sale at Thong Lo,53872040.0,197.087552,Baania
15761,BN2067,Condo for sale at สาทร,53609574.0,198.523740,Baania
23356,BN11717,Condo for sale at สีลม,53424938.0,197.066261,Baania
4895,DD5875,condominium for sale at สาทร,52830035.0,195.893695,DDProperty
7367,DD8840,คอนโด for sale at Sathorn,52491406.0,193.967195,DDProperty
8632,DD10349,คอนโด for sale at Sathorn,51321051.0,193.327591,DDProperty
5886,DD7058,Condo for sale at สุขุมวิท,50777556.0,199.565391,DDProperty
2559,DD3088,Condo for sale at Silom,50642338.0,190.611685,DDProperty
19485,BN6750,condominium for sale at Sukhumvit,50490034.0,199.392950,Baania
16942,BN3539,condominium for sale at Sathorn,50300000.0,196.849517,Baania


## 6. บันทึกรายงานการรัน

pipeline.py คุมลำดับ generation → cleaning → transformation → training แล้วบันทึก JSON report; notebook นี้เก็บรูปแบบรายงานเดียวกัน และเพิ่ม audit รายขั้นกับ metadata สำหรับอ้างอิงผลที่รันจริง

แหล่งโค้ด: [src/pipeline.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/pipeline.py)

In [22]:
report['total_time'] = time.perf_counter() - pipeline_started
with open(RUN_DIR/'data/pipeline_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)
pd.DataFrame(audit).to_csv(RUN_DIR/'data/notebook_step_audit.csv', index=False)
run_metadata = {'repository_commit': 'cf4c4affdc43d3f1450fe67efcd8185c224a72c4',
                'mock_data': GENERATE_MOCK, 'seed': 42, 'run_date': datetime.now().isoformat(),
                'pandas': pd.__version__, 'numpy': np.__version__, 'sklearn': sklearn.__version__}
with open(RUN_DIR/'data/notebook_run_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(run_metadata, f, ensure_ascii=False, indent=2)
print(json.dumps(report, ensure_ascii=False, indent=2))
print('Created files:')
for path in sorted(RUN_DIR.rglob('*')):
    if path.is_file():
        print(path.relative_to(RUN_DIR))

{
  "generation_time": 0.869384645000082,
  "cleaning": {
    "before": 50350,
    "after": 43028,
    "operations": {
      "deduplication": 2021,
      "missing_values": 1322,
      "spam_outliers": 3979
    }
  },
  "cleaning_time": 1.1576475949999576,
  "transformation": {
    "final_record_count": 42525
  },
  "transformation_time": 3.101736181999968,
  "modeling": {
    "Linear Regression": {
      "MAE": 24329407.230510846,
      "RMSE": 41363272.246085234,
      "R2": 0.5177437470558688
    },
    "Random Forest": {
      "MAE": 15282283.813541688,
      "RMSE": 34747675.327983715,
      "R2": 0.6596704963063325
    },
    "Gradient Boosting": {
      "MAE": 15886341.842155231,
      "RMSE": 32858278.75422912,
      "R2": 0.6956749445664051
    }
  },
  "modeling_time": 18.572567857999957,
  "total_time": 25.584297075999984
}
Created files:
data/cleaned/cleaned_listings.csv
data/notebook_run_metadata.json
data/notebook_step_audit.csv
data/pipeline_report.json
data/raw/baania.cs

## 7. ผลเดิมที่บันทึกใน repo และสิ่งที่ต้องแยกจากผลใหม่

ตารางต่อไปนี้มาจาก pipeline_report.json ของ commit ที่อ่าน ไม่ใช่ผลใหม่ของ notebook: เริ่ม 50,350 แถว หลัง cleaning 43,028 แถว หลัง transformation 42,525 แถว; ลบซ้ำ 2,021, missing 1,322, spam/outliers 3,979 แถว เวลารันเดิมประมาณ 11.10 วินาที ไม่ใช่เวลาที่รับประกันของเครื่องอื่น

| โมเดล | MAE (THB) | RMSE (THB) | R² เดิม |
|---|---:|---:|---:|
| Linear Regression | 24,329,801.47 | 41,363,221.66 | 0.517745 |
| Random Forest | 15,346,600.00 | 35,141,130.30 | 0.651920 |
| Gradient Boosting | 15,827,279.48 | 32,797,575.84 | 0.696798 |

ผลรันใหม่อาจต่างเพราะวันที่และ dependency versions การเลือกโมเดลต้นฉบับใช้ R² สูงสุด ไม่ใช่ MAE ต่ำสุด

### จุดที่ README และโค้ดไม่ตรงกัน

| ประเด็น | สิ่งที่โค้ดทำจริง |
|---|---|
| รวบรวมจากเว็บไซต์ | pipeline หลักเรียก mock generator; scraper เป็น template |
| Fuzzy deduplication | drop_duplicates เทียบ id และ title/location/price_raw ตรงกัน |
| กรองด้วย IQR | ใช้ค่าคงที่และ percentile 99 ของ price_per_sqm |
| ทำนายราคาจริง | ผลประเมินมาจากข้อมูลจำลองและ preprocessing ตามที่อธิบาย |

### พจนานุกรมข้อมูลสุดท้าย

| คอลัมน์ | ความหมาย |
|---|---|
| id | รหัสประกาศจากแหล่งจำลอง |
| title | หัวข้อหลัง cleanup |
| price_thb | ราคา THB หลังแปลงและเติมค่าว่าง |
| area_sqm | พื้นที่หน่วย ตร.ม. |
| price_per_sqm | price_thb / area_sqm |
| bedrooms / bathrooms | จำนวนห้องนอน / ห้องน้ำ |
| property_type | ประเภทหลัง normalize |
| location | ทำเลหลัง normalize |
| zone | โซนจาก mapping ใน source |
| district | สำเนาของ location |
| floor | ชั้นอาคารหรือ 0 เมื่อแปลงไม่ได้ |
| near_bts | ข้อมูลรถไฟฟ้าที่มีใน raw source |
| has_bts | 1 ถ้ามีข้อความ near_bts; 0 ถ้าไม่มี |
| posted_date | ข้อความวันที่ดิบ |
| age_days | อายุประกาศตามวันที่รัน หรือ 180 เมื่อ parse ไม่ได้ |
| source | ชื่อแหล่งข้อมูลจำลอง |
| price_imputed | flag ของการเติมราคาตาม logic ต้นฉบับ |


## แหล่งอ้างอิงโค้ด

- [README.md](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/README.md)
- [src/pipeline.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/pipeline.py)
- [src/extraction/mock_generator.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/extraction/mock_generator.py)
- [src/extraction/scraper_template.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/extraction/scraper_template.py)
- [src/cleaning/cleaner.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/cleaning/cleaner.py)
- [src/transformation/transformer.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/transformation/transformer.py)
- [src/modeling/predictor.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/src/modeling/predictor.py)
- [app/server.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/app/server.py)
- [tests/test_pipeline.py](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/tests/test_pipeline.py)
- [data/pipeline_report.json](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/data/pipeline_report.json)
- [models/evaluation.json](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/models/evaluation.json)
- [Pumpkin_Grill_Data_Pipeline.ipynb](https://github.com/chanidapa14092547/real-estate-aggregator/blob/cf4c4affdc43d3f1450fe67efcd8185c224a72c4/Pumpkin_Grill_Data_Pipeline.ipynb)

Notebook เดิมใน repo รวมโค้ดเป็น 4 ขั้นใหญ่ ไฟล์นี้ขยายคำอธิบายและแยกการเรียกแต่ละ method เพื่อดูการจัดการข้อมูลระหว่างขั้นครบขึ้น